# 03 ScienceWorld CPU feasibility
Run on CPU. This notebook installs ScienceWorld, runs a short task trajectory twice, and compares observations. It does not generate reversibility labels. Matching observation text is not proof that the full underlying state matches.

Run sections in order. Stop at an error and share the traceback. No models, tokens, or paid APIs are needed. Existing sandbox files are unchanged.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "scienceworld"], check=True)


## Connect storage and record the installed environment

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import json, importlib.metadata, platform, subprocess, inspect, time
drive.mount("/content/drive")
RUN = Path("/content/drive/MyDrive/ADNN_Irreversibility/results") / ("scienceworld_pilot_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
RUN.mkdir(parents=True, exist_ok=False)
versions = {"scienceworld":importlib.metadata.version("scienceworld"), "python":sys.version, "java":subprocess.run(["java","-version"], capture_output=True,text=True,check=True).stderr.strip()}
(RUN / "versions.json").write_text(json.dumps(versions,indent=2))
print(json.dumps(versions,indent=2))


## Start the simulator and inspect supported interfaces
No simplification preset is enabled. The API manifest is saved so we can design state comparison against the installed version.

In [ ]:
import os
option = "-Xlog:os+container=off"
if option not in os.environ.get("JAVA_TOOL_OPTIONS", "").split():
    os.environ["JAVA_TOOL_OPTIONS"] = (os.environ.get("JAVA_TOOL_OPTIONS", "") + " " + option).strip()
from scienceworld import ScienceWorldEnv
env = ScienceWorldEnv(envStepLimit=30)
api = {}
for name in dir(type(env)):
    if name.startswith("_"): continue
    member = getattr(type(env), name)
    if callable(member):
        try: api[name] = str(inspect.signature(member))
        except (ValueError,TypeError): api[name] = "signature unavailable"
(RUN / "api_manifest.json").write_text(json.dumps(api,indent=2))
tasks = env.get_task_names()
assert tasks, "Simulator returned no task names."
TASK = "boil" if "boil" in tasks else tasks[0]
print("Task count:",len(tasks))
print("Selected task:",TASK)
print("State-related methods:", [x for x in api if any(word in x.lower() for word in ["state", "tree", "object", "save", "restore"])])


## Run a short trajectory and replay it
The first run chooses up to three valid actions, preferring navigation or opening an object. The second run uses the exact recorded action sequence. This is a limited observation-replay test, not a full-state or reversibility test.

In [ ]:
def start_task():
    env.load(TASK, 0, "")
    observation, info = env.reset()
    return observation, info

initial, initial_info = start_task()
actions, first_trace = [], []
first_trace.append({"observation":initial})
started = time.perf_counter()
for index in range(3):
    candidates = sorted(set(env.get_valid_action_object_combinations()))
    if not candidates: break
    preferred = [a for a in candidates if a.startswith(("open ", "go to ", "move to "))]
    action = (preferred or candidates)[0]
    observation, reward, done, info = env.step(action)
    actions.append(action)
    first_trace.append({"action":action,"observation":observation,"reward":float(reward),"done":bool(done)})
    if done: break

replayed_initial, _ = start_task()
second_trace = [{"observation":replayed_initial}]
for action in actions:
    observation, reward, done, info = env.step(action)
    second_trace.append({"action":action,"observation":observation,"reward":float(reward),"done":bool(done)})
    if done: break

summary = {"scienceworld_version":versions["scienceworld"],"task":TASK,"variation":0,"simplifications":"", "actions":actions,"steps_replayed":len(second_trace)-1,"observation_trace_matches":first_trace == second_trace,"elapsed_seconds":round(time.perf_counter()-started,3),"full_state_equality_validated":False,"saved_to":str(RUN)}
(RUN / "traces.json").write_text(json.dumps({"first":first_trace,"replay":second_trace},indent=2))
(RUN / "summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))


## Close the Java simulator
Run once after saving the summary. If running the experiment again later, recreate the environment using section 3.

In [ ]:
env.close()
print("ScienceWorld simulator closed. Pilot files saved.")


Send the summary and the state-related method list back to the project chat. A true observation_trace_matches result only establishes replay for this short trajectory. Next we inspect full-state access, define meaningful state and partial restoration, and test restoration search before producing labels.

References: https://github.com/allenai/ScienceWorld and https://github.com/allenai/ScienceWorld/blob/main/scienceworld/scienceworld.py
